<a href="https://colab.research.google.com/github/Enrik-Shabani/ML-Project-GroupA/blob/main/notebooks/03_Supervised_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3 — Supervised Learning
## Beijing Multi-Site Air Quality · HEC Lausanne ML Project

**Reproducibility** : Dataset downloaded automatically from Google Drive — run all cells.

**Input** : `data_clean_featured.csv` (Notebook 1)  
**Target** : `PM2.5` (µg/m³)  
**Models** : OLS Baseline · Ridge Regression · Histogram Gradient Boosting

---

1. Setup & Data Loading

In [1]:
!pip install -q gdown --upgrade

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

COLOR_RIDGE  = '#1565C0'
COLOR_GB     = '#C62828'
COLOR_BASE   = '#757575'
RANDOM_STATE = 42

print('Libraries loaded.')

Libraries loaded.


In [ ]:
import gdown

FILE_ID     = '1T9gq-MBUJyZHIW8uMIRPzfCxNslLov3j'
OUTPUT_PATH = 'data_clean_featured.csv'

gdown.download(id=FILE_ID, output=OUTPUT_PATH, quiet=False)

df = pd.read_csv(OUTPUT_PATH)
print(f'Dataset loaded — shape: {df.shape}')
df.head(3)

Downloading...
From (original): https://drive.google.com/uc?id=1T9gq-MBUJyZHIW8uMIRPzfCxNslLov3j
From (redirected): https://drive.google.com/uc?id=1T9gq-MBUJyZHIW8uMIRPzfCxNslLov3j&confirm=t&uuid=5f733a16-1e5f-44bd-91a1-5f79672e6862
To: /Users/fey/Desktop/Projects HEC/ML/group_A/ML-Project-GroupA/notebooks/data_clean_featured.csv
 16%|██████▎                                 | 25.2M/158M [01:27<08:13, 269kB/s]

In [ ]:
# ── Diagnostic Overview ──────────────────────────────────────────────────────

# 1. Shape
print("Shape:", df.shape)

# 2. Column names
print("\nColumns:", df.columns.tolist())

# 3. Data types
print("\nData types:")
print(df.dtypes)

# 4. Date range
print("\nDate range:", df["datetime"].min(), "→", df["datetime"].max())

# 5. Number of stations
print("\nStations:", df["station"].nunique(), "→", sorted(df["station"].unique()))

# 6. Missing values
missing = df.isnull().sum()
missing = missing[missing > 0]
print("\nMissing values:")
print(missing if not missing.empty else "  None")

# 7. Sorted by station and datetime?
is_sorted = (df[["station", "datetime"]]
             .equals(df[["station", "datetime"]].sort_values(["station", "datetime"])))
print("\nSorted by station + datetime:", is_sorted)

# 8. PM2.5 summary
print("\nPM2.5 summary:")
print(df["PM2.5"].describe().round(2))


In [ ]:
# ── Supervised Feature Configurations ────────────────────────────────────────

target   = "PM2.5"
all_cols = df.columns.tolist()

# ── Column-group discovery ────────────────────────────────────────────────────

# PM2.5 lag / rolling features
lag_cols     = [c for c in all_cols if "lag"  in c.lower() and ("2.5" in c or "pm2" in c.lower())]
rolling_cols = [c for c in all_cols if "roll" in c.lower() and ("2.5" in c or "pm2" in c.lower())]

# Cyclical encodings
all_cyc_cols  = [c for c in all_cols if c.endswith("_sin") or c.endswith("_cos")]
wind_dir_cols = [c for c in all_cyc_cols if "wind" in c.lower() or "wd" in c.lower()]
time_cyc_cols = [c for c in all_cyc_cols if c not in wind_dir_cols]

# Station dummies, weather, extra temporal
station_cols     = [c for c in all_cols if c.startswith("station_")]
weather_cols     = [c for c in all_cols if c in {"TEMP", "PRES", "DEWP", "RAIN", "WSPM"}]
temporal_cols    = [c for c in all_cols if c in {"season", "day_of_week"}]
other_pollutants = [c for c in all_cols if c in {"SO2", "NO2", "CO", "O3"}]

# Columns to always exclude
always_exclude = {"PM2.5", "PM10", "datetime", "station", "wd"}
if time_cyc_cols:                          # cyclical encodings exist → drop raw date parts
    always_exclude |= {"year", "month", "day", "hour"}

# ── Feature sets ──────────────────────────────────────────────────────────────

# A: weather + temporal + cyclical time + wind direction + station dummies
features_A = sorted(set(
    weather_cols + temporal_cols + time_cyc_cols + wind_dir_cols + station_cols
) - always_exclude)

# B: PM2.5 lag and rolling mean features only
features_B = sorted(set(lag_cols + rolling_cols) - always_exclude)

# C: A + other pollutants (SO2, NO2, CO, O3) + lag/rolling features
features_C = sorted(set(
    weather_cols + temporal_cols + time_cyc_cols + wind_dir_cols +
    station_cols + other_pollutants + lag_cols + rolling_cols
) - always_exclude)

# ── Feature counts and lists ──────────────────────────────────────────────────

print("=" * 60)
print(f"Target : {target}")
print("=" * 60)

for name, fset in [("A", features_A), ("B", features_B), ("C", features_C)]:
    print(f"\nFeature set {name}  ({len(fset)} features):")
    for f in fset:
        print(f"  {f}")

# ── Sanity checks ─────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("Sanity checks")
print("=" * 60)

for name, fset in [("A", features_A), ("B", features_B), ("C", features_C)]:
    pm25_in = "PM2.5" in fset
    pm10_in = "PM10"  in fset
    missing = [f for f in fset if f not in df.columns]
    print(f"\nSet {name}:")
    print(f"  PM2.5 excluded : {'FAIL - PM2.5 present!' if pm25_in else 'pass'}")
    print(f"  PM10  excluded : {'FAIL - PM10 present!'  if pm10_in else 'pass'}")
    print(f"  All cols in df : {'FAIL - missing: ' + str(missing) if missing else 'pass'}")

In [ ]:
# ── Temporal Train / Test Split ───────────────────────────────────────────────
# We split by time, not randomly. A random split would leak future observations
# into training, making the model appear far more accurate than it will be on
# genuinely unseen future data. Keeping the last 20% of time as the test set
# mirrors real deployment: the model is always predicting forward in time.

# 1. Sort by datetime
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)

# 2. Find the cutoff timestamp at the 80% row position, then split on a clean
#    time boundary so no single hour is split across train and test
#    (multiple stations share identical timestamps — a row-index cut would
#     place the same hour in both sets and break the strict ordering check).
cutoff_time = df["datetime"].iloc[int(len(df) * 0.80)]
train_df    = df[df["datetime"] <  cutoff_time].copy()
test_df     = df[df["datetime"] >= cutoff_time].copy()

# 3. Shapes
print("Shapes")
print(f"  train_df : {train_df.shape}")
print(f"  test_df  : {test_df.shape}")

# 4. Date ranges
print("\nDate ranges")
print(f"  train : {train_df['datetime'].min()}  →  {train_df['datetime'].max()}")
print(f"  test  : {test_df['datetime'].min()}  →  {test_df['datetime'].max()}")

# 5. Verify no overlap
assert test_df["datetime"].min() > train_df["datetime"].max(), \
    "FAIL: test set overlaps with training set!"
print("\nTemporal integrity check : pass  (test starts after train ends)")

In [ ]:
# ── Common Evaluation Function ────────────────────────────────────────────────

def evaluate_model(model_name, feature_set_name, model, features,
                   train_df, test_df, target):
    X_train = train_df[features].copy()
    y_train = train_df[target]
    X_test  = test_df[features].copy()
    y_test  = test_df[target]

    # Encode any remaining object columns (e.g. season, day_of_week as strings)
    obj_cols = X_train.select_dtypes(include="object").columns.tolist()
    if obj_cols:
        X_train = pd.get_dummies(X_train, columns=obj_cols, drop_first=True)
        X_test  = pd.get_dummies(X_test,  columns=obj_cols, drop_first=True)
        # Align test to train columns in case a category is absent in one split
        X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)

    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred  = model.predict(X_test)

    return {
        "model":       model_name,
        "feature_set": feature_set_name,
        "train_rmse":  round(mean_squared_error(y_train, y_train_pred) ** 0.5, 4),
        "test_rmse":   round(mean_squared_error(y_test,  y_test_pred)  ** 0.5, 4),
        "train_mae":   round(mean_absolute_error(y_train, y_train_pred), 4),
        "test_mae":    round(mean_absolute_error(y_test,  y_test_pred),  4),
        "train_r2":    round(r2_score(y_train, y_train_pred), 4),
        "test_r2":     round(r2_score(y_test,  y_test_pred),  4),
    }

results = []

print("evaluate_model() defined — results list initialised.")

### OLS Baseline

In [ ]:
# ── OLS Baseline ──────────────────────────────────────────────────────────────

from sklearn.linear_model import LinearRegression

for feat_name, features in [("A", features_A), ("B", features_B), ("C", features_C)]:
    results.append(evaluate_model(
        model_name      = "OLS",
        feature_set_name= feat_name,
        model           = LinearRegression(),
        features        = features,
        train_df        = train_df,
        test_df         = test_df,
        target          = target,
    ))
    print(f"OLS [{feat_name}] done.")

# ── Results so far ────────────────────────────────────────────────────────────

results_df = pd.DataFrame(results)
results_df

### Ridge Regression

In [ ]:
# ── Ridge Regression ─────────────────────────────────────────────────────────

from sklearn.pipeline import Pipeline

alphas = np.logspace(-2, 4, 30)          # 0.01 … 10 000, log-spaced
tss    = TimeSeriesSplit(n_splits=5)

for feat_name, features in [("A", features_A), ("B", features_B), ("C", features_C)]:
    ridge_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge",  RidgeCV(alphas=alphas, cv=tss)),
    ])

    results.append(evaluate_model(
        model_name       = "Ridge",
        feature_set_name = feat_name,
        model            = ridge_pipe,
        features         = features,
        train_df         = train_df,
        test_df          = test_df,
        target           = target,
    ))

    best_alpha = ridge_pipe.named_steps["ridge"].alpha_
    print(f"Ridge [{feat_name}] done — best alpha: {best_alpha:.4g}")

# ── Results so far ────────────────────────────────────────────────────────────

results_df = pd.DataFrame(results)
results_df

**Why Ridge B and C are identical to OLS.**  
`RidgeCV` selected **α = 0.01** (the smallest value in the grid) for both feature sets B and C. With N = 336 552 training rows, the regularisation term α·‖β‖² is negligible relative to the data term — the effective ratio α/N ≈ 3 × 10⁻⁸. At that strength, Ridge solves essentially the same normal equations as OLS, producing coefficients — and therefore predictions — that are indistinguishable to four decimal places.  
This is expected behaviour, not a bug: when a dataset is large relative to the number of features and the features are already informative (lag/rolling means), OLS is already well-determined and regularisation adds nothing. Feature set A selects α = 356, confirming that Ridge *does* regularise when the features are weaker and the model is closer to underdetermined.

### Gradient Boosting

In [ ]:
# ── Histogram Gradient Boosting ───────────────────────────────────────────────

hgb_params = dict(
    max_iter        = 500,
    learning_rate   = 0.05,
    max_depth       = 6,
    min_samples_leaf= 50,
    l2_regularization= 0.1,
    random_state    = RANDOM_STATE,
)

for feat_name, features in [("A", features_A), ("B", features_B), ("C", features_C)]:
    results.append(evaluate_model(
        model_name       = "HGB",
        feature_set_name = feat_name,
        model            = HistGradientBoostingRegressor(**hgb_params),
        features         = features,
        train_df         = train_df,
        test_df          = test_df,
        target           = target,
    ))
    print(f"HGB [{feat_name}] done.")

# ── Full results sorted by test RMSE ──────────────────────────────────────────

results_df = pd.DataFrame(results).sort_values("test_rmse").reset_index(drop=True)
results_df

In [ ]:
# ── Supervised Results Comparison ────────────────────────────────────────────

col_order = ["model", "feature_set",
             "train_rmse", "test_rmse", "rmse_gap",
             "train_mae",  "test_mae",
             "train_r2",   "test_r2"]

results_df = (
    pd.DataFrame(results)
    .assign(rmse_gap=lambda d: round(d["test_rmse"] - d["train_rmse"], 4))
    .sort_values("test_rmse")
    .reset_index(drop=True)
)[col_order]

best_idx  = results_df["test_rmse"].idxmin()
num_cols  = results_df.select_dtypes("number").columns.tolist()

def highlight_best(row):
    color = "background-color: #d4edda" if row.name == best_idx else ""
    return [color] * len(row)

display(
    results_df.style
    .apply(highlight_best, axis=1)
    .format({c: "{:.4f}" for c in num_cols})
    .set_caption("Model comparison — sorted by test RMSE  |  green row = best")
)

best     = results_df.iloc[0]
gap_pct  = best["rmse_gap"] / best["test_rmse"] * 100
print(f"Best model : {best['model']}  (feature set {best['feature_set']})")
print(f"Test RMSE  : {best['test_rmse']:.4f}    Test R² : {best['test_r2']:.4f}")
print(f"RMSE gap   : {best['rmse_gap']:.4f}  ({gap_pct:.1f}% of test RMSE)")

### Interpretation

**Best model.** HGB with feature set C achieves the lowest test RMSE (17.04 µg/m³) and the highest test R² (0.958), narrowly beating OLS C and Ridge C (both 17.41). The non-linear interactions captured by gradient boosting — between co-pollutants, lag values, and weather — give it a small but consistent edge over the linear models.

**Feature set contribution.** The most striking result is the gap between feature set A (weather/temporal only) and sets B/C. All three model families drop from a test RMSE of ~65 to ~17–19 once PM2.5 lag and rolling-mean features are added (set B). This confirms that recent PM2.5 history is by far the strongest predictor. Adding co-pollutants on top (set C) provides an incremental improvement of roughly 1–2 RMSE units, most visible for HGB.

**Overfitting.** OLS and Ridge show near-zero train–test RMSE gaps across all feature sets, indicating low variance and well-constrained models. HGB C has a gap of 2.00 (≈ 11.7 % of test RMSE), which is acceptable given the dataset size. HGB A shows the largest gap (7.50, ≈ 13.2 %), reflecting the model overfitting moderately when forced to learn from weak features alone — a sign that regularisation or early stopping could be tightened if feature set A were used in isolation.

### Model comparison plot

In [ ]:
# ── Model Comparison Plot ─────────────────────────────────────────────────────

plot_df = (
    results_df
    .assign(label=lambda d: d["model"] + "-" + d["feature_set"])
    .sort_values("test_rmse")
    .reset_index(drop=True)
)

best_label = plot_df.loc[plot_df["test_rmse"].idxmin(), "label"]
colors = [COLOR_GB if lbl == best_label else COLOR_BASE for lbl in plot_df["label"]]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(plot_df["label"], plot_df["test_rmse"], color=colors, width=0.6, edgecolor="white")

# Annotate the best bar
best_pos = plot_df["test_rmse"].idxmin()
best_val = plot_df.loc[best_pos, "test_rmse"]
ax.annotate(
    f"Best\n{best_val:.2f} µg/m³",
    xy=(best_pos, best_val),
    xytext=(best_pos, best_val + 3.5),
    ha="center", va="bottom", fontsize=9, color=COLOR_GB, fontweight="bold",
    arrowprops=dict(arrowstyle="-", color=COLOR_GB, lw=1.2),
)

ax.set_title("Test RMSE by model and feature set  (lower is better)", fontsize=13, pad=12)
ax.set_xlabel("Model – Feature set", fontsize=11)
ax.set_ylabel("Test RMSE  (µg/m³)", fontsize=11)
ax.set_xticks(range(len(plot_df)))
ax.set_xticklabels(plot_df["label"], rotation=35, ha="right", fontsize=10)
ax.set_ylim(0, plot_df["test_rmse"].max() * 1.20)

plt.tight_layout()
plt.show()

### Store fitted models for interpretation

In [ ]:
# ── Store Fitted Models ───────────────────────────────────────────────────────
# Refit each model with identical hyperparameters so the fitted objects are
# available for interpretation (feature importance, prediction diagnostics, residual analysis, etc.).
# results_df and all metrics from previous cells are not modified.

from sklearn.linear_model import LinearRegression

model_configs = [
    ("OLS",   lambda: LinearRegression()),
    ("Ridge", lambda: Pipeline([
                  ("scaler", StandardScaler()),
                  ("ridge",  RidgeCV(alphas=np.logspace(-2, 4, 30),
                                     cv=TimeSeriesSplit(n_splits=5))),
              ])),
    ("HGB",   lambda: HistGradientBoostingRegressor(
                  max_iter=500, learning_rate=0.05, max_depth=6,
                  min_samples_leaf=50, l2_regularization=0.1,
                  random_state=RANDOM_STATE,
              )),
]

feature_sets = [("A", features_A), ("B", features_B), ("C", features_C)]

trained_models = {}

for model_name, model_factory in model_configs:
    for feat_name, features in feature_sets:
        X_train = train_df[features].copy()
        y_train = train_df[target]

        obj_cols = X_train.select_dtypes(include="object").columns.tolist()
        if obj_cols:
            X_train = pd.get_dummies(X_train, columns=obj_cols, drop_first=True)

        m = model_factory()
        m.fit(X_train, y_train)
        trained_models[(model_name, feat_name)] = m
        print(f"  fitted and stored  ({model_name}, {feat_name!r})")

print("\ntrained_models keys:")
for k in sorted(trained_models):
    print(f"  {k}")

assert ("HGB", "C") in trained_models, "HGB-C not found!"
print(f"\ntrained_models[('HGB', 'C')] : {type(trained_models[('HGB', 'C')]).__name__}  — OK")

### Lightweight HGB-C hyperparameter tuning

In [ ]:
# ── HGB-C Lightweight Hyperparameter Tuning ───────────────────────────────────

# 1. Encode features_C — identical logic to evaluate_model
X_tr = train_df[features_C].copy()
y_tr = train_df[target]
X_te = test_df[features_C].copy()
y_te = test_df[target]

obj_cols = X_tr.select_dtypes(include="object").columns.tolist()
if obj_cols:
    X_tr = pd.get_dummies(X_tr, columns=obj_cols, drop_first=True)
    X_te = pd.get_dummies(X_te, columns=obj_cols, drop_first=True)
    X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)

# 2. Small grid — 2 values per param → 16 candidates × 3 CV splits = 48 fits
param_grid = {
    "learning_rate":     [0.05, 0.10],
    "max_iter":          [300,  500],
    "max_leaf_nodes":    [31,   63],
    "l2_regularization": [0.0,  0.5],
}

tss_tune = TimeSeriesSplit(n_splits=3)

n_candidates = 1
for v in param_grid.values():
    n_candidates *= len(v)

gs = GridSearchCV(
    HistGradientBoostingRegressor(min_samples_leaf=50, random_state=RANDOM_STATE),
    param_grid,
    cv=tss_tune,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1,
    refit=True,
)

print(f"Fitting {n_candidates} candidates × {tss_tune.n_splits} CV splits …")
gs.fit(X_tr, y_tr)

print("\nBest parameters:")
for k, v in gs.best_params_.items():
    print(f"  {k:22s}: {v}")
print(f"\nBest CV RMSE (train folds) : {-gs.best_score_:.4f}")

# 3. Evaluate the refitted best estimator on the held-out test set
tuned = gs.best_estimator_   # refit=True → already fitted on full X_tr

y_tr_pred = tuned.predict(X_tr)
y_te_pred = tuned.predict(X_te)

train_rmse_t = round(mean_squared_error(y_tr, y_tr_pred) ** 0.5, 4)
test_rmse_t  = round(mean_squared_error(y_te, y_te_pred) ** 0.5, 4)

tuned_row = {
    "model":       "HGB_tuned",
    "feature_set": "C",
    "train_rmse":  train_rmse_t,
    "test_rmse":   test_rmse_t,
    "rmse_gap":    round(test_rmse_t - train_rmse_t, 4),
    "train_mae":   round(mean_absolute_error(y_tr, y_tr_pred), 4),
    "test_mae":    round(mean_absolute_error(y_te, y_te_pred),  4),
    "train_r2":    round(r2_score(y_tr, y_tr_pred), 4),
    "test_r2":     round(r2_score(y_te, y_te_pred),  4),
}

# 4. Drop any previous HGB_tuned-C row so re-running this cell stays idempotent
results_df = results_df[
    ~((results_df["model"] == "HGB_tuned") & (results_df["feature_set"] == "C"))
].reset_index(drop=True)
results_df = pd.concat([results_df, pd.DataFrame([tuned_row])], ignore_index=True)

# 5. Store fitted model
trained_models[("HGB_tuned", "C")] = tuned
print(f"\ntrained_models[('HGB_tuned', 'C')] stored — {type(tuned).__name__}")

# 6. Compare original HGB-C vs tuned — both lookups pin model AND feature_set
compare_cols = ["model", "feature_set", "train_rmse", "test_rmse", "rmse_gap", "test_r2"]
mask = (
    ((results_df["model"] == "HGB")       & (results_df["feature_set"] == "C")) |
    ((results_df["model"] == "HGB_tuned") & (results_df["feature_set"] == "C"))
)
compare_df = (results_df.loc[mask, compare_cols]
              .sort_values("test_rmse")
              .reset_index(drop=True))

print("\n── HGB-C vs HGB_tuned-C ─────────────────────────────────────")
print(compare_df.to_string(index=False))

orig_rmse  = results_df.loc[
    (results_df["model"] == "HGB") & (results_df["feature_set"] == "C"), "test_rmse"
].values[0]
tuned_rmse = results_df.loc[
    (results_df["model"] == "HGB_tuned") & (results_df["feature_set"] == "C"), "test_rmse"
].values[0]
delta = orig_rmse - tuned_rmse
print(f"\n  Test RMSE improvement from tuning : {delta:+.4f} µg/m³  "
      f"({'improved' if delta > 0 else 'no gain'})")

### Final Supervised Model Selection

**Tuning procedure.**  
Hyperparameter search for HGB-C used GridSearchCV with TimeSeriesSplit(n_splits=3) on the training set,
keeping the held-out test period strictly unseen throughout.

**Tuning outcome.**  
The best configuration found by the grid search (`learning_rate=0.05`, `max_iter=300`, `max_leaf_nodes=31`, `l2_regularization=0.0`)
yielded a test RMSE of **17.08 µg/m³**, marginally *worse* than the original HGB-C (17.04 µg/m³).
The difference (−0.04 µg/m³) is negligible and within noise.

**Decision.**  
Because the tuned variant offers no improvement, the original HGB-C model is retained as the final supervised model.
This result confirms that the initial hyperparameters were already competitive within the tested search space —
further grid expansion is unlikely to yield meaningful gains given the plateau observed here.

**Final model.**  
All subsequent interpretation (feature importance, SHAP, residual analysis) will use `trained_models[("HGB", "C")]`.

#
#
#

F
e
a
t
u
r
e

I
m
p
o
r
t
a
n
c
e

—

F
i
n
a
l

S
u
p
e
r
v
i
s
e
d

M
o
d
e
l

(
H
G
B
-
C
)

In [ ]:
# ── Permutation Feature Importance — HGB-C (test set) ────────────────────────

FINAL_MODEL = trained_models[("HGB", "C")]

# Encode features_C with the same logic used during training
X_tr_enc = train_df[features_C].copy()
X_te_enc = test_df[features_C].copy()
y_te     = test_df[target]

obj_cols = X_tr_enc.select_dtypes(include="object").columns.tolist()
if obj_cols:
    X_tr_enc = pd.get_dummies(X_tr_enc, columns=obj_cols, drop_first=True)
    X_te_enc = pd.get_dummies(X_te_enc, columns=obj_cols, drop_first=True)
    X_te_enc = X_te_enc.reindex(columns=X_tr_enc.columns, fill_value=0)

# Permutation importance on the test set (RMSE-based scoring)
perm = permutation_importance(
    FINAL_MODEL, X_te_enc, y_te,
    scoring="neg_root_mean_squared_error",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

imp_df = (
    pd.DataFrame({
        "feature":    X_te_enc.columns,
        "importance": perm.importances_mean,
        "std":        perm.importances_std,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

top15 = imp_df.head(15)

print("Top 15 features by permutation importance (mean RMSE increase when permuted):")
print(top15[["feature", "importance", "std"]].to_string(index=False))

# ── Plot ──────────────────────────────────────────────────────────────────────
COPOLLUTANTS = {"NO2", "CO", "SO2", "O3"}

def feat_color(f):
    if "pm25" in f.lower():  return COLOR_GB
    if f in COPOLLUTANTS:    return COLOR_RIDGE
    return COLOR_BASE

colors = [feat_color(f) for f in top15["feature"]]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(
    top15["feature"][::-1],
    top15["importance"][::-1],
    xerr=top15["std"][::-1],
    color=colors[::-1],
    edgecolor="white",
    height=0.6,
    error_kw=dict(elinewidth=0.8, capsize=3, ecolor="#555"),
)

from matplotlib.patches import Patch
legend_handles = [
    Patch(color=COLOR_GB,    label="PM2.5 lag / rolling"),
    Patch(color=COLOR_RIDGE, label="Co-pollutants"),
    Patch(color=COLOR_BASE,  label="Weather / time / station"),
]
ax.legend(handles=legend_handles, fontsize=9, loc="lower right")

ax.set_xlabel("Mean RMSE increase when feature is permuted  (µg/m³)", fontsize=11)
ax.set_title("Permutation Feature Importance — HGB-C  (test set, top 15)", fontsize=13, pad=12)
ax.axvline(0, color="black", linewidth=0.7, linestyle="--")
plt.tight_layout()
plt.show()

#### Interpretation

**Short-term PM2.5 memory is the dominant signal.**  
The permutation importance analysis shows that `pm25_lag_1h` is by far the most important feature: permuting it produces the largest increase in RMSE. This means that the model relies heavily on the most recent PM2.5 concentration to predict the current level. `pm25_roll_3h` and `pm25_lag_3h` also contribute, but their importance is much smaller than the 1-hour lag. This confirms that PM2.5 has strong short-term temporal persistence.

**Co-pollutants provide additional predictive information.**  
Among the non-PM2.5 features, `CO`, `NO2`, `SO2`, and `O3` appear in the top 15. Their importance is clearly lower than `pm25_lag_1h`, but they still add useful information to the model. This is consistent with the comparison between feature sets B and C: adding co-pollutants and contextual variables improves the prediction slightly compared with using lag and rolling PM2.5 features alone.

**Weather and time variables have secondary importance.**  
Meteorological and temporal variables such as `DEWP`, `TEMP`, `hour_sin`, `hour_cos`, and `WSPM` appear with smaller importance values. This suggests that they help refine predictions, but they are not the main drivers of model performance. Their effect is likely partly absorbed by the lag features, which already summarize the recent atmospheric state.

**Station effects are limited in the final model.**  
Only one station dummy appears near the bottom of the top 15, indicating that spatial differences between monitoring stations play a smaller role once recent PM2.5 history, co-pollutants, and weather conditions are included. Overall, the model mainly predicts PM2.5 from short-term temporal persistence, then refines predictions using pollution and weather context.

### Prediction Diagnostics — Final Supervised Model (HGB-C)


In [ ]:
# ── Prediction Diagnostics — HGB-C (test set) ─────────────────────

from sklearn.metrics import mean_absolute_error, r2_score

FINAL_MODEL = trained_models[("HGB", "C")]

# ── Encode features_C — same logic as training ──────────────────────
X_tr_enc = train_df[features_C].copy()
X_te_enc = test_df[features_C].copy()
y_te     = test_df[target].values

obj_cols = X_tr_enc.select_dtypes(include="object").columns.tolist()
if obj_cols:
    X_tr_enc = pd.get_dummies(X_tr_enc, columns=obj_cols, drop_first=True)
    X_te_enc = pd.get_dummies(X_te_enc, columns=obj_cols, drop_first=True)
    X_te_enc = X_te_enc.reindex(columns=X_tr_enc.columns, fill_value=0)

y_pred = FINAL_MODEL.predict(X_te_enc)

# ── Test-set metrics ─────────────────────────────────────────
rmse = np.sqrt(((y_te - y_pred) ** 2).mean())
mae  = mean_absolute_error(y_te, y_pred)
r2   = r2_score(y_te, y_pred)

print(f"Test RMSE : {rmse:.4f} µg/m³")
print(f"Test MAE  : {mae:.4f} µg/m³")
print(f"Test R²   : {r2:.4f}")

# ── Line plot — first 1 000 observations ────────────────────────────
N = min(1000, len(y_te))

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(range(N), y_te[:N],   color=COLOR_BASE, linewidth=0.9,
        label="Actual PM2.5",    alpha=0.85)
ax.plot(range(N), y_pred[:N], color=COLOR_GB,   linewidth=0.9,
        label="Predicted PM2.5", alpha=0.85)
ax.set_xlabel("Test observation index", fontsize=11)
ax.set_ylabel("PM2.5 (µg/m³)", fontsize=11)
ax.set_title("Actual vs Predicted PM2.5 — HGB-C  (first 1 000 test obs.)", fontsize=13, pad=10)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# ── Scatter plot — predicted vs actual ───────────────────────────────
lim_lo = min(float(y_te.min()), float(y_pred.min())) - 5
lim_hi = max(float(y_te.max()), float(y_pred.max())) * 1.05

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_te, y_pred, alpha=0.15, s=6, color=COLOR_GB, rasterized=True)
ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi],
        color="black", linewidth=1.2, linestyle="--", label="y = x")
ax.set_xlabel("Actual PM2.5 (µg/m³)", fontsize=11)
ax.set_ylabel("Predicted PM2.5 (µg/m³)", fontsize=11)
ax.set_title("Predicted vs Actual PM2.5 — HGB-C  (full test set)", fontsize=13, pad=10)
ax.set_xlim(lim_lo, lim_hi)
ax.set_ylim(lim_lo, lim_hi)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()


#### Interpretation

**The model captures the overall temporal pattern.**  
The time-series plot shows that the HGB-C model follows the main rises and falls of PM2.5 during the test period. Predicted values generally move in the same direction as observed values, which confirms that the model captures the dominant short-term dynamics of air pollution.

**Predictions are strongest for low-to-moderate PM2.5 levels.**  
The predicted-versus-actual scatter plot shows a dense concentration of points around the `y = x` line, especially for the most common PM2.5 values. This indicates good predictive accuracy for normal and moderately polluted conditions, which represent the majority of the test observations.

**Extreme pollution episodes remain harder to predict.**  
For very high PM2.5 values, the scatter becomes more dispersed and several points fall below the diagonal. This suggests that the model tends to underpredict some extreme peaks. These events are rarer and more difficult to capture because they may depend on sudden local emissions, unusual weather conditions, or short-lived atmospheric changes not fully represented in the available features.

**Overall diagnostic conclusion.**  
The visual diagnostics are consistent with the numerical results: HGB-C provides strong general predictive performance, but its errors increase during rare high-pollution episodes. The model is therefore reliable for general PM2.5 forecasting patterns, while extreme spikes should be interpreted with more caution.

### Residual Analysis — Final Supervised Model (HGB-C)


In [ ]:
# ── Residual Analysis — HGB-C (test set) ─────────────────────────────────

FINAL_MODEL = trained_models[("HGB", "C")]

# ── Encode features_C — same logic as training ─────────────────────
X_tr_enc = train_df[features_C].copy()
X_te_enc = test_df[features_C].copy()
y_te     = test_df[target].values

obj_cols = X_tr_enc.select_dtypes(include="object").columns.tolist()
if obj_cols:
    X_tr_enc = pd.get_dummies(X_tr_enc, columns=obj_cols, drop_first=True)
    X_te_enc = pd.get_dummies(X_te_enc, columns=obj_cols, drop_first=True)
    X_te_enc = X_te_enc.reindex(columns=X_tr_enc.columns, fill_value=0)

y_pred    = FINAL_MODEL.predict(X_te_enc)
residuals = y_te - y_pred

# ── Summary statistics ────────────────────────────────────────────────
print("Residual summary  (actual − predicted):")
print(f"  Mean   : {residuals.mean():.4f} µg/m³")
print(f"  Median : {np.median(residuals):.4f} µg/m³")
print(f"  Std    : {residuals.std():.4f} µg/m³")
print(f"  Min    : {residuals.min():.4f} µg/m³")
print(f"  Max    : {residuals.max():.4f} µg/m³")
print(f"  P5     : {np.percentile(residuals, 5):.4f} µg/m³")
print(f"  P95    : {np.percentile(residuals, 95):.4f} µg/m³")

# ── Histogram of residuals ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(residuals, bins=80, color=COLOR_GB, edgecolor="white", linewidth=0.3, alpha=0.85)
ax.axvline(0,              color="black",     linewidth=1.4, linestyle="--",
           label="zero")
ax.axvline(residuals.mean(), color=COLOR_RIDGE, linewidth=1.2, linestyle="-",
           label=f"mean ({residuals.mean():.2f} µg/m³)")
ax.set_xlabel("Residual (actual − predicted, µg/m³)", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
ax.set_title("Residual Distribution — HGB-C  (test set)", fontsize=13, pad=10)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# ── Residuals vs predicted ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(y_pred, residuals, alpha=0.15, s=5, color=COLOR_GB, rasterized=True)
ax.axhline(0, color="black", linewidth=1.2, linestyle="--")
ax.set_xlabel("Predicted PM2.5 (µg/m³)", fontsize=11)
ax.set_ylabel("Residual (actual − predicted, µg/m³)", fontsize=11)
ax.set_title("Residuals vs Predicted PM2.5 — HGB-C  (test set)", fontsize=13, pad=10)
plt.tight_layout()
plt.show()


#### Interpretation

**Residuals are centered close to zero.**  
The mean residual is approximately 0.21 µg/m³ and the median is approximately -0.03 µg/m³. This indicates that the model has no strong global bias: on average, predictions are neither systematically too high nor systematically too low.

**Most errors are small, but extreme errors remain.**  
The 5th and 95th percentiles are around -20.14 and +20.88 µg/m³, meaning that most residuals remain within a relatively narrow range. However, the minimum and maximum residuals are much larger in magnitude, showing that the model can still make large errors during rare extreme pollution situations.

**Error variance increases at higher predicted PM2.5 levels.**  
The residuals-versus-predicted plot shows that the residual cloud is tighter for low predicted PM2.5 values and becomes wider as predicted PM2.5 increases. This suggests heteroscedasticity: the model is more stable for low-to-moderate pollution levels and less precise during high-pollution episodes.

**Extreme pollution episodes are the main limitation.**  
Large positive residuals correspond to cases where actual PM2.5 is much higher than predicted, meaning the model underpredicts some severe peaks. Large negative residuals also occur, meaning some high predictions overestimate the actual value. Overall, the model performs well for the majority of observations, but rare and abrupt pollution spikes remain harder to predict accurately.

### Partial Dependence Plots — Final Supervised Model (HGB-C)


In [ ]:
# ── Partial Dependence Plots — HGB-C ────────────────────────────────────

from sklearn.inspection import PartialDependenceDisplay

FINAL_MODEL = trained_models[("HGB", "C")]

# ── Encode features_C — same logic as training ─────────────────────
X_tr_enc = train_df[features_C].copy()

obj_cols = X_tr_enc.select_dtypes(include="object").columns.tolist()
if obj_cols:
    X_tr_enc = pd.get_dummies(X_tr_enc, columns=obj_cols, drop_first=True)

# ── Sample 5 000 training rows for speed ────────────────────────────
rng    = np.random.default_rng(42)
idx    = rng.choice(len(X_tr_enc), size=min(5000, len(X_tr_enc)), replace=False)
X_samp = X_tr_enc.iloc[idx].reset_index(drop=True)

# ── Continuous features to plot (station dummies excluded) ──────────
PDP_FEATURES = [f for f in ["pm25_lag_1h", "pm25_roll_3h", "CO", "NO2", "TEMP"]
                if f in X_samp.columns]
print(f"PDP features: {PDP_FEATURES}")

# ── Plot — one PDP per axis ──────────────────────────────────────────
n = len(PDP_FEATURES)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
axes = np.array(axes).ravel()

for ax, feat in zip(axes, PDP_FEATURES):
    PartialDependenceDisplay.from_estimator(
        FINAL_MODEL, X_samp,
        features=[feat],
        kind="average",
        ax=ax,
        line_kw={"color": COLOR_GB, "linewidth": 1.8},
        n_jobs=-1,
    )
    ax.set_title(feat, fontsize=11)
    ax.set_xlabel(feat, fontsize=10)
    ax.set_ylabel("Predicted PM2.5 (µg/m³)", fontsize=10)

fig.suptitle(
    "Partial Dependence Plots — HGB-C  (training sample, n = 5 000)",
    fontsize=13,
)
plt.tight_layout()
plt.show()


#### Interpretation

**The 1-hour PM2.5 lag has the clearest positive effect.**  
The PDP for `pm25_lag_1h` shows a strong, almost monotonic positive relationship with predicted PM2.5. When the previous hour’s PM2.5 concentration increases, the model predicts a much higher current PM2.5 level. This is consistent with the permutation importance results, where `pm25_lag_1h` was by far the most important feature.

**The 3-hour rolling PM2.5 effect is more complex.**  
Unlike `pm25_lag_1h`, the PDP for `pm25_roll_3h` decreases at higher values. This should be interpreted carefully because lag and rolling features are highly correlated. Once the model already knows the very recent PM2.5 value through `pm25_lag_1h`, the rolling mean may capture whether pollution is rising or falling relative to the immediate past. Therefore, the negative PDP does not mean that high PM2.5 history is generally protective; rather, it reflects the model’s marginal association after averaging over other correlated temporal features.

**Co-pollutants are positively associated with predicted PM2.5.**  
The PDPs for `CO` and `NO2` show increasing relationships with predicted PM2.5. Higher levels of these co-pollutants are associated with higher predicted PM2.5, which is plausible because they can reflect shared emission sources such as traffic, combustion, and urban pollution episodes.

**Temperature has a negative marginal association in the model.**  
The PDP for `TEMP` decreases as temperature increases, suggesting that, within this model, higher temperatures are associated with lower predicted PM2.5 when averaged over the other features. This effect is weaker than the temporal PM2.5 signal and should be interpreted cautiously, because weather variables interact with season, atmospheric stability, dispersion, and pollution chemistry.

**PDPs show model-learned associations, not causal effects.**  
Partial dependence plots show how the model prediction changes when one feature varies while averaging over the observed distribution of other features. They are useful for interpretation, but they do not prove causal mechanisms, especially when predictors are strongly correlated, as is the case for lag and rolling PM2.5 features.

## Supervised Learning — Conclusion

**Task.**  
This notebook addressed a supervised regression problem: predicting hourly PM2.5 concentrations at twelve Beijing monitoring stations using weather, temporal, station, co-pollutant, lag, and rolling features. A strict temporal train/test split was used to evaluate the model on future observations rather than on randomly mixed data.

**Feature engineering mattered more than model complexity.**  
Three feature sets were evaluated across all model families. Feature set A, based on weather, time, wind direction, season, and station variables, produced the weakest results, showing that contextual variables alone are insufficient to predict PM2.5 accurately at hourly resolution. Feature set B, based only on PM2.5 lag and rolling features, already achieved strong performance, confirming strong short-term temporal persistence. Feature set C, which combined contextual variables, co-pollutants, and PM2.5 lag/rolling features while excluding PM10, performed best overall across the model families.

**Final model.**  
HGB-C, meaning Histogram Gradient Boosting with feature set C, was selected as the final supervised model. It achieved a test RMSE of approximately **17.04 µg/m³** and a test R² of approximately **0.958**, explaining roughly 96% of the variance in held-out hourly PM2.5 concentrations. OLS-C and Ridge-C reached comparable performance, indicating that the gain from non-linear modelling was modest once strong engineered features were available. This suggests that feature engineering, especially the inclusion of recent PM2.5 history, contributed more to predictive accuracy than model complexity alone.

**Hyperparameter tuning.**  
A lightweight GridSearchCV search over HGB-C hyperparameters, including learning rate, number of iterations, leaf nodes, and L2 regularisation, did not improve the test RMSE compared with the initial HGB-C configuration. The tuned model was marginally worse by about **0.04 µg/m³**, which is negligible. Therefore, the original HGB-C model was retained as the final supervised model.

**Feature importance and model interpretation.**  
Permutation importance showed that `pm25_lag_1h` was by far the dominant predictor, confirming that the most recent PM2.5 concentration is the strongest signal for current PM2.5. Other short-term lag and rolling features also contributed, but less strongly. Co-pollutants such as CO, NO2, SO2, and O3 added useful secondary information, while weather, time, and station variables played smaller roles. The Partial Dependence Plots supported this interpretation: `pm25_lag_1h` showed a clear positive association with predicted PM2.5, while `pm25_roll_3h` showed a more complex marginal pattern because lag and rolling variables are highly correlated. CO and NO2 were positively associated with predicted PM2.5, while temperature showed a weaker negative marginal association. These patterns should be interpreted as model-learned associations, not causal effects.

**Prediction and residual diagnostics.**  
The prediction plots showed that HGB-C captures the general temporal dynamics of PM2.5 well, especially for low-to-moderate pollution levels. Residuals were approximately centered around zero, with a mean close to zero and no strong global bias. However, the residual spread increased at higher predicted PM2.5 values, indicating heteroscedasticity. Large residuals occurred on both sides, meaning that the model can both underpredict and overpredict rare extreme pollution episodes. These high-pollution events remain the main prediction challenge.

**Limitations and outlook.**  
All results are predictive and associative, not causal. The model learns statistical regularities from the observed period, so performance may decrease under future pollution regimes that differ from the training period. The temporal split reduces look-ahead bias, but it does not eliminate the risk of distribution shift over longer horizons. Improving peak-pollution prediction would likely require additional physical and external drivers, such as boundary layer height, wind-trajectory data, regional emissions, or methods designed to better model the tails of the PM2.5 distribution, such as quantile regression.